# Week 3 — Rule-Based Baseline Agents & Tournament
**SoC 2026 | Dynamic Pricing Using Reinforcement Learning**

**Topics Covered:**
- Implementing 4 rule-based agents: Random, AlwaysNash, AlwaysCollude, TitForTat
- Axelrod tournament setup (round-robin, 1000+ rounds)
- Results tables and profit plots
- Collusion Index calculation

**Resources studied (SOC Week 3 PDF):**
- Crash Course: Prisoner's Dilemma & Cooperation
- William Spaniel: Tit-for-Tat & Repeated Games
- Nicky Case: Evolution of Trust (ncase.me/trust) — ALL 3 ACTS
- Axelrod (1980, 1984) — Evolution of Cooperation Ch. 1 & 2
- Calvano et al. Section 2 (re-read with agents in mind)

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from itertools import combinations

from src.environment.pricing_env import BertrandPricingEnv
from src.agents.random_agent import RandomAgent
from src.agents.always_nash import AlwaysNashAgent
from src.agents.always_collude import AlwaysColludeAgent
from src.agents.tit_for_tat import TitForTatAgent
from src.utils.helper import run_tournament, plot_price_history, plot_profit_comparison, collusion_index

In [ ]:
# Setup environment
env = BertrandPricingEnv(a=10, b=2, c=0.5, mc=1.0, p_max=8.0, n_prices=50, seed=42)

print(f"Nash price    : {env.p_nash:.4f}")
print(f"Collude price : {env.p_collude:.4f}")
print(f"Marginal cost : {env.mc:.4f}")

## 1. Agent Descriptions

| Agent | Strategy | Expected Behaviour |
|---|---|---|
| **Random** | Uniform random price | Chaotic — weakest baseline |
| **AlwaysNash** | Always P_nash | Competitive — drives market to NE |
| **AlwaysCollude** | Always P_collude | High prices — profitable IF opponent also colludes |
| **TitForTat** | Mirror opponent's last action | Nice, retaliating, forgiving, clear (Axelrod 1984) |

In [ ]:
# Instantiate all agents
agents = {
    "Random":       RandomAgent(env.n_prices, seed=42),
    "AlwaysNash":   AlwaysNashAgent(env),
    "AlwaysCollude": AlwaysColludeAgent(env),
    "TitForTat":    TitForTatAgent(env),
}

for name, agent in agents.items():
    print(f"{name:15} | action sample: {agent.act()}")

## 2. Round-Robin Tournament (1000 rounds per matchup)

> ⚠️ **SOC Pitfall (Week 3):** Run 1000+ rounds so defection incentives emerge. Always set random seeds.

In [ ]:
N_ROUNDS = 1000
agent_names = list(agents.keys())
matchups = list(combinations(agent_names, 2))

tournament_results = {}
rows = []

print(f"Running round-robin tournament ({N_ROUNDS} rounds per matchup)...\n")

for name_a, name_b in matchups:
    env_t = BertrandPricingEnv(a=10, b=2, c=0.5, mc=1.0, p_max=8.0,
                                n_prices=50, max_steps=N_ROUNDS, seed=42)
    agent_a = agents[name_a]
    agent_b = agents[name_b]

    # Reset agents
    agent_a.reset()
    agent_b.reset()

    result = run_tournament(env_t, agent_a, agent_b, n_rounds=N_ROUNDS, seed=42)
    key = f"{name_a} vs {name_b}"
    tournament_results[key] = result

    s = result['summary']
    ci_a = collusion_index(s['avg_price_a'], s['p_nash'], s['p_collude'])
    ci_b = collusion_index(s['avg_price_b'], s['p_nash'], s['p_collude'])

    rows.append({
        'Matchup': key,
        'Avg Price A': round(s['avg_price_a'], 3),
        'Avg Price B': round(s['avg_price_b'], 3),
        'Avg Profit A': round(s['avg_profit_a'], 3),
        'Avg Profit B': round(s['avg_profit_b'], 3),
        'Total Profit A': round(s['total_profit_a'], 1),
        'Total Profit B': round(s['total_profit_b'], 1),
        'Collusion Index A': round(ci_a, 3),
        'Collusion Index B': round(ci_b, 3),
    })
    print(f"{key}: Profit A={s['avg_profit_a']:.3f} | Profit B={s['avg_profit_b']:.3f} | CI_A={ci_a:.3f}")

df = pd.DataFrame(rows)
print("\nTournament Complete!")

In [ ]:
# Display results table
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)
print("\n" + "=" * 80)
print("TOURNAMENT RESULTS TABLE")
print("=" * 80)
print(df[['Matchup', 'Avg Profit A', 'Avg Profit B', 'Collusion Index A', 'Collusion Index B']].to_string(index=False))

# Save to CSV
df.to_csv('../results/tables/week3_tournament_results.csv', index=False)
print("\nTable saved → results/tables/week3_tournament_results.csv")

## 3. Price Trajectory Plots

In [ ]:
# Plot price histories for all matchups
fig, axes = plt.subplots(3, 2, figsize=(16, 14))
axes = axes.flatten()

for idx, (key, result) in enumerate(tournament_results.items()):
    ax = axes[idx]
    name_a, name_b = key.split(' vs ')
    last200 = 200

    ax.plot(result['prices_a'][-last200:], label=name_a, alpha=0.8)
    ax.plot(result['prices_b'][-last200:], label=name_b, alpha=0.8)
    ax.axhline(env.p_nash,    color='red',   linestyle='--', linewidth=1, label=f"Nash={env.p_nash:.2f}")
    ax.axhline(env.p_collude, color='green', linestyle='--', linewidth=1, label=f"Collude={env.p_collude:.2f}")
    ax.set_title(key)
    ax.set_xlabel("Round (last 200)")
    ax.set_ylabel("Price")
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)

# Hide unused subplot if odd number of matchups
for i in range(len(tournament_results), len(axes)):
    axes[i].set_visible(False)

plt.suptitle("Round-Robin Tournament — Price Trajectories (Last 200 Rounds)", fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig("../results/figures/week3_price_trajectories.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved → results/figures/week3_price_trajectories.png")

## 4. Profit Comparison Bar Chart

In [ ]:
plot_profit_comparison(tournament_results, env,
                       save_path="../results/figures/week3_profit_comparison.png")

## 5. Collusion Index Summary

In [ ]:
print("COLLUSION INDEX SUMMARY")
print("Collusion Index: 0 = Nash (competitive), 1 = Full Collusion")
print("-" * 55)
for _, row in df.iterrows():
    avg_ci = (row['Collusion Index A'] + row['Collusion Index B']) / 2
    print(f"{row['Matchup']:30} | Avg CI = {avg_ci:.3f}")

print()
print("Key Observations:")
print("- AlwaysCollude vs AlwaysCollude → CI ≈ 1.0 (full collusion)")
print("- AlwaysNash vs AlwaysNash       → CI ≈ 0.0 (fully competitive)")
print("- TitForTat can sustain cooperation IF opponent also plays nicely")
print("- Random has CI ≈ 0.5 (mixed — no strategic intent)")

## 6. Week 3 Summary

| Agent | Avg Profit | Collusion Index | Notes |
|---|---|---|---|
| Random | Low | ~0.5 | Chaotic, unpredictable |
| AlwaysNash | Medium | ~0.0 | Drives market to NE |
| AlwaysCollude | High (if unopposed) | ~1.0 | Exploitable by defectors |
| TitForTat | Highest sustained | ~0.7–0.9 | Best overall — Axelrod's winner |

**Baselines are ready.** The Q-Learning agent (Week 4) must beat Random as the minimum gate.

**Next Week:** Implement tabular Q-learning from scratch.